# 04: Knowledge Graph

Phase 2. Extracts cross-references from the clause corpus and builds a NetworkX graph. Runs a hop-count experiment to justify `GRAPH_HOPS=2`. Saves `data/cross_refs.jsonl`.

Run after notebook 01 (`data/clauses.jsonl` must exist).

## Cell group 1: Imports

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from collections import Counter
import networkx as nx

SCRIPTS_DIR = Path("scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from cross_refs import extract_cross_refs, refs_to_records
from graph_build import load_clauses, load_cross_refs, build_graph, graph_stats, neighbours

# Paths
DATA_DIR  = Path("data")
XREF_PATH = DATA_DIR / "cross_refs.jsonl"

# Graph retriever config
GRAPH_HOPS = 2  # set by hop-count experiment (Group 6); update before moving to notebook 05

print(f"NetworkX {nx.__version__}  |  scripts at {SCRIPTS_DIR}")
print(f"XREF_PATH  : {XREF_PATH}")
print(f"GRAPH_HOPS : {GRAPH_HOPS}")


In [ ]:
from log_setup import setup_logging

logger = setup_logging("04_knowledge_graph")

## Cell group 2: Load corpus

In [3]:
clauses = load_clauses()
clause_index = {c["clause_id"]: c for c in clauses}

print(f"Clauses : {len(clauses)}  |  unique IDs : {len(clause_index)}")
print()
for src, n in Counter(c["source"] for c in clauses).most_common():
    print(f"  {src:<20} {n}")

Clauses : 2568  |  unique IDs : 2568

  jmlsg_2              901
  poca_2002            764
  jmlsg_1              567
  mlr_2017             218
  fatf_40              69
  fca_fcg              49


## Cell group 3: Extract cross-references

If `data/cross_refs.jsonl` already exists this step is skipped: the file is treated as the authoritative output and reloaded directly.

In [4]:
# Staleness guard: warn if clauses.jsonl was updated after cross_refs.jsonl was written.
# If this fires, delete cross_refs.jsonl and re-run the cell below to force re-extraction.
if XREF_PATH.exists():
    xref_mtime = os.path.getmtime(XREF_PATH)
    clause_mtime = os.path.getmtime(DATA_DIR / "clauses.jsonl")
    if clause_mtime > xref_mtime:
        print("WARNING: clauses.jsonl is newer than cross_refs.jsonl")
        print("  The cross-refs may have been built from a different version of the corpus.")
        print("  Delete cross_refs.jsonl and re-run the extraction cell to force rebuild.")
    else:
        print("cross_refs.jsonl is up to date (newer than clauses.jsonl)")
else:
    print("cross_refs.jsonl does not exist: will be created by the extraction cell below.")


cross_refs.jsonl is up to date (newer than clauses.jsonl)


In [5]:
if XREF_PATH.exists():
    xref_records = load_cross_refs(XREF_PATH)
    print(f"Loaded {len(xref_records)} cross-refs from existing {XREF_PATH}")
    print("Delete cross_refs.jsonl and re-run this cell to force re-extraction.")
else:
    print("Extracting cross-references from clause text...")
    all_refs = []
    for clause in clauses:
        all_refs.extend(extract_cross_refs(clause, clause_index))

    resolved = [r for r in all_refs if r.target_id is not None]
    unresolved = [r for r in all_refs if r.target_id is None]

    print(f"Total refs  : {len(all_refs)}")
    print(f"Resolved    : {len(resolved)}")
    print(f"Unresolved  : {len(unresolved)}  (repealed sections / out-of-corpus, not an error)")

    xref_records = refs_to_records(resolved)
    with XREF_PATH.open("w", encoding="utf-8") as f:
        for r in xref_records:
            f.write(json.dumps(r) + "\n")
    print(f"Written {len(xref_records)} resolved cross-refs to {XREF_PATH}")

Loaded 2302 cross-refs from existing data\cross_refs.jsonl
Delete cross_refs.jsonl and re-run this cell to force re-extraction.


In [6]:
# Breakdown by source and sample
src_counts = Counter(r["source_id"].rsplit("_", 1)[0] for r in xref_records)
print("Resolved cross-refs by source:")
for src, n in src_counts.most_common():
    print(f"  {src:<30} {n}")

print("\nSample resolved refs:")
for r in xref_records[:8]:
    print(f"  {r['source_id']:<35} --[{r['target_raw']}]--> {r['target_id']}")

Resolved cross-refs by source:
  poca_2002                      1799
  mlr_2017_reg                   175
  jmlsg_2                        101
  mlr_2017_sch7                  77
  fatf_40                        67
  jmlsg_1                        41
  mlr_2017_sch6                  37
  mlr_2017_sch4                  5

Sample resolved refs:
  mlr_2017_reg_3                      --[regulation 55(2)]--> mlr_2017_reg_55
  mlr_2017_reg_3                      --[regulation 14(1)]--> mlr_2017_reg_14
  mlr_2017_reg_3                      --[regulation 31(4)]--> mlr_2017_reg_31
  mlr_2017_reg_3                      --[regulation 11]--> mlr_2017_reg_11
  mlr_2017_reg_3                      --[regulation 5]--> mlr_2017_reg_5
  mlr_2017_reg_3                      --[regulation 6]--> mlr_2017_reg_6
  mlr_2017_reg_3                      --[regulation 4]--> mlr_2017_reg_4
  mlr_2017_reg_3                      --[regulation 34(4)]--> mlr_2017_reg_34


## Cell group 4: Build NetworkX graph

Rebuilt from `clauses.jsonl` + `cross_refs.jsonl` at notebook startup (~100ms). This is the graph used for retrieval experiments.

In [7]:
G = build_graph(clauses, cross_refs=xref_records)
stats = graph_stats(G)

print(f"Nodes : {stats['total_nodes']}")
print(f"Edges : {stats['total_edges']}")
print()
print("Node types:")
for t, n in sorted(stats["node_types"].items()):
    print(f"  {t:<15} {n}")
print()
print("Edge types:")
for t, n in sorted(stats["edge_types"].items()):
    print(f"  {t:<25} {n}")

Nodes : 2826
Edges : 5299

Node types:
  clause          2568
  document        6
  term            252

Edge types:
  CROSS_REFERENCES          2302
  DEFINES                   429
  IN_DOCUMENT               2568


In [8]:
# Sample defined terms
defined = [
    (u, G.nodes[v]["name"])
    for u, v, d in G.edges(data=True)
    if d.get("rel") == "DEFINES"
]
print(f"DEFINES edges: {len(defined)}")
print("Sample:")
for cid, term in defined[:8]:
    print(f"  {cid:<35} defines '{term}'")

DEFINES edges: 429
Sample:
  mlr_2017_reg_3                      defines 'annex 1 financial institution'
  mlr_2017_reg_3                      defines 'appropriate body'
  mlr_2017_reg_3                      defines 'auction platform'
  mlr_2017_reg_3                      defines 'authorised person'
  mlr_2017_reg_3                      defines 'the fca'
  mlr_2017_reg_3                      defines 'bill payment service provider'
  mlr_2017_reg_3                      defines 'business relationship'
  mlr_2017_reg_3                      defines 'the capital requirements directive'


## Cell group 5: Traversal smoke test

In [9]:
# Quick sanity check before the hop experiment
seeds = ["poca_2002_s327", "mlr_2017_reg_28", "mlr_2017_reg_33"]

for seed in seeds:
    if seed not in G:
        print(f"  {seed}: NOT IN GRAPH (check clause_id format)")
        continue
    hop1 = neighbours(G, seed, rel="CROSS_REFERENCES", hops=1)
    print(f"{seed}  ({G.nodes[seed].get('marker')})")
    print(f"  1-hop neighbours: {hop1[:6]}")

poca_2002_s327  (Section 327)
  1-hop neighbours: ['poca_2002_s338', 'poca_2002_s339A']
mlr_2017_reg_28  (Regulation 28)
  1-hop neighbours: ['mlr_2017_reg_17', 'mlr_2017_reg_19', 'mlr_2017_reg_5', 'mlr_2017_reg_18', 'mlr_2017_reg_27', 'mlr_2017_reg_31']
mlr_2017_reg_33  (Regulation 33)
  1-hop neighbours: ['mlr_2017_reg_17', 'mlr_2017_reg_29', 'mlr_2017_reg_18', 'mlr_2017_reg_34', 'mlr_2017_reg_35', 'mlr_2017_reg_28']


## Cell group 6: Hop-count experiment

Sweeps hops 1–3 over a set of seed clauses. Records:
- How many distinct neighbours are retrieved at each hop count
- How many new clauses each additional hop adds (marginal gain)
- Overlap between hops (are we just getting noise at hop 3?)

Use this to pick the hop count for the hybrid retriever (notebook 05).

In [10]:
# Seed clauses covering different document types
experiment_seeds = [
    "poca_2002_s327",   # concealment: central money laundering offence
    "poca_2002_s330",   # failure to disclose
    "mlr_2017_reg_28",  # CDD measures
    "mlr_2017_reg_33",  # enhanced due diligence
    "mlr_2017_reg_35",  # enhanced measures: high-risk third countries
]

# Filter to seeds actually in the graph
experiment_seeds = [s for s in experiment_seeds if s in G]
print(f"Seeds in graph: {experiment_seeds}")

Seeds in graph: ['poca_2002_s327', 'poca_2002_s330', 'mlr_2017_reg_28', 'mlr_2017_reg_33', 'mlr_2017_reg_35']


In [11]:
rows = []
for seed in experiment_seeds:
    hop1 = set(neighbours(G, seed, rel="CROSS_REFERENCES", hops=1))
    hop2 = set(neighbours(G, seed, rel="CROSS_REFERENCES", hops=2))
    hop3 = set(neighbours(G, seed, rel="CROSS_REFERENCES", hops=3))

    rows.append({
        "seed": seed,
        "hop1": len(hop1),
        "hop2": len(hop2),
        "hop2_new": len(hop2 - hop1),         # clauses added by going to hop 2
        "hop3": len(hop3),
        "hop3_new": len(hop3 - hop2),         # clauses added by going to hop 3
    })

print(f"{'seed':<35} {'h1':>4} {'h2':>4} {'h2+':>5} {'h3':>4} {'h3+':>5}")
print("-" * 60)
for r in rows:
    print(f"{r['seed']:<35} {r['hop1']:>4} {r['hop2']:>4} {r['hop2_new']:>5} {r['hop3']:>4} {r['hop3_new']:>5}")

seed                                  h1   h2   h2+   h3   h3+
------------------------------------------------------------
poca_2002_s327                         2    2     0    2     0
poca_2002_s330                         1    6     5   18    12
mlr_2017_reg_28                        6   10     4   16     6
mlr_2017_reg_33                        6   14     8   18     4
mlr_2017_reg_35                        4    8     4   15     7


In [12]:
# Corpus-wide average: how many clauses does each hop count retrieve on average?
all_clause_ids = [
    n for n, d in G.nodes(data=True)
    if d.get("node_type") == "clause"
]

# Sample 50 random clause seeds for a representative average
random.seed(42)
sample_seeds = random.sample(all_clause_ids, min(50, len(all_clause_ids)))

totals = {1: 0, 2: 0, 3: 0}
for seed in sample_seeds:
    for h in (1, 2, 3):
        totals[h] += len(neighbours(G, seed, rel="CROSS_REFERENCES", hops=h))

n = len(sample_seeds)
print(f"Average neighbours retrieved (sample of {n} seeds):")
for h, total in totals.items():
    print(f"  hops={h} : {total/n:.1f} clauses")
print()
print("Rule of thumb: pick the smallest hop count that retrieves enough context.")
print("hops=2 is usually the best tradeoff: 1 often misses cross-document refs, 3 adds noise.")

Average neighbours retrieved (sample of 50 seeds):
  hops=1 : 0.9 clauses
  hops=2 : 3.2 clauses
  hops=3 : 7.2 clauses

Rule of thumb: pick the smallest hop count that retrieves enough context.
hops=2 is usually the best tradeoff — 1 often misses cross-document refs, 3 adds noise.


In [13]:
# GRAPH_HOPS is set in cell 2. Review the hop table above before changing it.
print(f"Graph retriever will use hops={GRAPH_HOPS}")
print("Update GRAPH_HOPS in cell 2 after evaluating retrieval quality.")


Graph retriever will use hops=2
Update GRAPH_HOPS in cell 2 after evaluating retrieval quality.


## Cell group 7: Completion check

In [ ]:
xref_count = sum(1 for _ in XREF_PATH.open(encoding="utf-8")) if XREF_PATH.exists() else 0
clause_node_count = sum(1 for _, d in G.nodes(data=True) if d.get("node_type") == "clause")

print("=== Completion check ===")
print(f"cross_refs.jsonl exists     : {XREF_PATH.exists()}")
print(f"Resolved cross-refs written : {xref_count}  (need >= 100)")
print(f"Graph clause nodes          : {clause_node_count}  (should = {len(clauses)})")
print(f"Graph total edges           : {G.number_of_edges()}")
print(f"GRAPH_HOPS set to           : {GRAPH_HOPS}")

In [18]:
# Graph invariant assertions: run after completion check to catch silent regressions.
# These fire on every kernel restart so problems are caught early, not in notebook 05.

stats = graph_stats(G)

assert stats["edge_types"].get("DEFINES", 0) > 0, \
    "No DEFINES edges: check graph_build.py quote patterns or clause text encoding"

assert stats["edge_types"].get("CROSS_REFERENCES", 0) > 100, \
    f"Only {stats['edge_types'].get('CROSS_REFERENCES', 0)} CROSS_REFERENCES: re-extract cross_refs.jsonl"

assert stats["node_types"].get("term", 0) > 0, \
    "No term nodes: DEFINES extraction produced no output"

assert G.has_edge("poca_2002_s327", "poca_2002_s338"), \
    "s327→s338 edge missing: cross_refs.jsonl may be stale or built from a different corpus version"

print("All graph invariants passed.")
print(f"  DEFINES edges   : {stats['edge_types']['DEFINES']}")
print(f"  CROSS_REFERENCES: {stats['edge_types']['CROSS_REFERENCES']}")
print(f"  term nodes      : {stats['node_types']['term']}")
print(f"  s327→s338 edge  : present")


All graph invariants passed.
  DEFINES edges   : 429
  CROSS_REFERENCES: 2302
  term nodes      : 252
  s327→s338 edge  : present
